# 05_train_baseline_convnext

This notebook trains the first clean baseline for the thesis.

## Experiment design

### Stage 1: base training
- initialize **ConvNeXt-Tiny** with **ImageNet pretrained weights**
- train on `train_base.csv` from **HAM**
- validate on `val_base.csv` from **HAM**

### Stage 2: target-domain adaptation
- load the best Stage 1 checkpoint
- fine-tune on `train_bcn_ft.csv` from **BCN**
- validate on `val_bcn_ft.csv` from **BCN**

### Final evaluation
- test on `test_bcn.csv`

## Current taxonomy
6 classes:
- `NV`
- `MEL`
- `BCC`
- `BKL`
- `AKIEC`
- `DF`

## Outputs
Saved under `../outputs/`:
- checkpoints
- metrics json files
- confusion matrix figure
- CSV with per-image BCN test predictions


## Notes

This notebook is intentionally simple and clean:
- first baseline only
- no saliency separation yet
- no segmentation masks yet
- no MixUp / CutMix
- moderate augmentations only

The goal is to get one stable end-to-end classifier before adding the thesis method.


In [12]:
from pathlib import Path
import os
import json
import copy
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as T

import timm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    recall_score,
)

import matplotlib.pyplot as plt

from tqdm.auto import tqdm


## Config


In [2]:
# -------------------------
# Paths
# -------------------------
SPLIT_DIR = Path("../data/processed/splits")
OUT_DIR = Path("../outputs")
CKPT_DIR = OUT_DIR / "checkpoints"
METRICS_DIR = OUT_DIR / "metrics"
FIG_DIR = OUT_DIR / "figures"
PRED_DIR = OUT_DIR / "predictions"

for p in [OUT_DIR, CKPT_DIR, METRICS_DIR, FIG_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_BASE_CSV = SPLIT_DIR / "train_base.csv"
VAL_BASE_CSV = SPLIT_DIR / "val_base.csv"
TRAIN_BCN_FT_CSV = SPLIT_DIR / "train_bcn_ft.csv"
VAL_BCN_FT_CSV = SPLIT_DIR / "val_bcn_ft.csv"
TEST_BCN_CSV = SPLIT_DIR / "test_bcn.csv"

# -------------------------
# Experiment
# -------------------------
SEED = 42
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("DEVICE:", DEVICE)
MODEL_NAME = "convnext_tiny"
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0  # safer default in notebooks
USE_WEIGHTED_SAMPLER = False  # start simple; class weights are already used in loss
LABEL_SMOOTHING = 0.05

# -------------------------
# Training schedule
# -------------------------
BASE_EPOCHS = 8
FT_EPOCHS = 6

BASE_LR = 3e-4
FT_LR = 1e-4
WEIGHT_DECAY = 1e-4

PRIMARY_CLASSES = ["NV", "MEL", "BCC", "BKL", "AKIEC", "DF"]
CLASS_TO_IDX = {c: i for i, c in enumerate(PRIMARY_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

print("DEVICE:", DEVICE)
print("MODEL_NAME:", MODEL_NAME)
print("IMAGE_SIZE:", IMAGE_SIZE)
print("PRIMARY_CLASSES:", PRIMARY_CLASSES)


DEVICE: mps
DEVICE: mps
MODEL_NAME: convnext_tiny
IMAGE_SIZE: 224
PRIMARY_CLASSES: ['NV', 'MEL', 'BCC', 'BKL', 'AKIEC', 'DF']


## Reproducibility


In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Seeded everything with:", SEED)


Seeded everything with: 42


## Load split CSVs


In [4]:
train_base_df = pd.read_csv(TRAIN_BASE_CSV)
val_base_df = pd.read_csv(VAL_BASE_CSV)
train_bcn_ft_df = pd.read_csv(TRAIN_BCN_FT_CSV)
val_bcn_ft_df = pd.read_csv(VAL_BCN_FT_CSV)
test_bcn_df = pd.read_csv(TEST_BCN_CSV)

print("train_base_df:", train_base_df.shape)
print("val_base_df:", val_base_df.shape)
print("train_bcn_ft_df:", train_bcn_ft_df.shape)
print("val_bcn_ft_df:", val_bcn_ft_df.shape)
print("test_bcn_df:", test_bcn_df.shape)


train_base_df: (9052, 32)
val_base_df: (2258, 32)
train_bcn_ft_df: (9625, 32)
val_bcn_ft_df: (3259, 32)
test_bcn_df: (3249, 32)


## Sanity checks on loaded splits


In [14]:
required_cols = ["image_path", "harmonized_label", "isic_id", "lesion_id"]

for name, df in [
    ("train_base", train_base_df),
    ("val_base", val_base_df),
    ("train_bcn_ft", train_bcn_ft_df),
    ("val_bcn_ft", val_bcn_ft_df),
    ("test_bcn", test_bcn_df),
]:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    missing_required = [c for c in required_cols if c not in df.columns]
    print("Missing required columns:", missing_required)

    print("Rows:", len(df))
    print("Unique lesions:", df["lesion_id"].nunique(dropna=True))
    print("Unique images:", df["isic_id"].nunique(dropna=True))

    print("Label counts:")
    display(
        df["harmonized_label"]
        .value_counts()
        .reindex(PRIMARY_CLASSES, fill_value=0)
        .to_frame("count")
    )

    missing_paths = df["image_path"].isna().sum()
    print("Missing image_path rows:", int(missing_paths))



train_base
Missing required columns: []
Rows: 9052
Unique lesions: 6834
Unique images: 9052
Label counts:


,count
harmonized_label,
NV,6195
MEL,1054
BCC,494
BKL,1061
AKIEC,120
DF,128


Missing image_path rows: 0

val_base
Missing required columns: []
Rows: 2258
Unique lesions: 1708
Unique images: 2258
Label counts:


,count
harmonized_label,
NV,1541
MEL,251
BCC,128
BKL,277
AKIEC,29
DF,32


Missing image_path rows: 0

train_bcn_ft
Missing required columns: []
Rows: 9625
Unique lesions: 2725
Unique images: 9625
Label counts:


...

Missing image_path rows: 0

val_bcn_ft
Missing required columns: []
Rows: 3259
Unique lesions: 909
Unique images: 3259
Label counts:


,count
harmonized_label,
NV,1172
MEL,748
BCC,762
BKL,319
AKIEC,217
DF,41


Missing image_path rows: 0

test_bcn
Missing required columns: []
Rows: 3249
Unique lesions: 909
Unique images: 3249
Label counts:


,count
harmonized_label,
NV,1155
MEL,819
BCC,699
BKL,317
AKIEC,231
DF,28


Missing image_path rows: 0


## Dataset and transforms


In [15]:
def safe_open_image(path):
    path = Path(path)
    with Image.open(path) as img:
        return img.convert("RGB")

train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

class SkinCSVClassificationDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None, return_metadata=False):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.return_metadata = return_metadata

        bad_labels = set(self.df["harmonized_label"].unique()) - set(self.class_to_idx.keys())
        if len(bad_labels) > 0:
            raise ValueError(f"Found unknown labels: {bad_labels}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_open_image(row["image_path"])
        if self.transform is not None:
            img = self.transform(img)

        y = self.class_to_idx[row["harmonized_label"]]

        if self.return_metadata:
            meta = {
                "isic_id": row["isic_id"],
                "lesion_id": row["lesion_id"],
                "image_path": row["image_path"],
                "label_str": row["harmonized_label"],
                "source_dataset": row.get("source_dataset", None),
            }
            return img, y, meta

        return img, y


## Class weights

We use class weights because the class distribution is imbalanced, especially in HAM.

This helps the loss penalize mistakes on rare classes more fairly.


In [16]:
def compute_class_weights_from_df(df, class_order):
    counts = df["harmonized_label"].value_counts().reindex(class_order, fill_value=0)
    counts = counts.astype(float)

    weights = 1.0 / counts
    weights = weights / weights.mean()

    return counts, weights

ham_counts, ham_weights = compute_class_weights_from_df(train_base_df, PRIMARY_CLASSES)
bcn_counts, bcn_weights = compute_class_weights_from_df(train_bcn_ft_df, PRIMARY_CLASSES)

print("HAM train counts:")
display(ham_counts.to_frame("count"))
print("HAM class weights:")
display(ham_weights.to_frame("weight"))

print("\nBCN fine-tune train counts:")
display(bcn_counts.to_frame("count"))
print("BCN class weights:")
display(bcn_weights.to_frame("weight"))


HAM train counts:


,count
harmonized_label,
NV,6195.0
MEL,1054.0
BCC,494.0
BKL,1061.0
AKIEC,120.0
DF,128.0


HAM class weights:


,weight
harmonized_label,
NV,0.047893
MEL,0.281494
BCC,0.600596
BKL,0.279637
AKIEC,2.472455
DF,2.317926



BCN fine-tune train counts:


,count
harmonized_label,
NV,3320.0
MEL,2436.0
BCC,2215.0
BKL,915.0
AKIEC,640.0
DF,99.0


BCN class weights:


,weight
harmonized_label,
NV,0.129834
MEL,0.176949
BCC,0.194604
BKL,0.471090
AKIEC,0.673511
DF,4.354013


## DataLoaders


In [17]:
def make_weighted_sampler(df, class_to_idx):
    labels = df["harmonized_label"].map(class_to_idx).values
    class_counts = np.bincount(labels, minlength=len(class_to_idx))
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    sample_weights = torch.tensor(sample_weights, dtype=torch.double)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    return sampler

train_base_ds = SkinCSVClassificationDataset(train_base_df, CLASS_TO_IDX, transform=train_transform)
val_base_ds = SkinCSVClassificationDataset(val_base_df, CLASS_TO_IDX, transform=eval_transform)

train_bcn_ft_ds = SkinCSVClassificationDataset(train_bcn_ft_df, CLASS_TO_IDX, transform=train_transform)
val_bcn_ft_ds = SkinCSVClassificationDataset(val_bcn_ft_df, CLASS_TO_IDX, transform=eval_transform)
test_bcn_ds = SkinCSVClassificationDataset(test_bcn_df, CLASS_TO_IDX, transform=eval_transform, return_metadata=True)

if USE_WEIGHTED_SAMPLER:
    train_base_sampler = make_weighted_sampler(train_base_df, CLASS_TO_IDX)
    train_bcn_ft_sampler = make_weighted_sampler(train_bcn_ft_df, CLASS_TO_IDX)
else:
    train_base_sampler = None
    train_bcn_ft_sampler = None

train_base_loader = DataLoader(
    train_base_ds,
    batch_size=BATCH_SIZE,
    shuffle=(train_base_sampler is None),
    sampler=train_base_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

val_base_loader = DataLoader(
    val_base_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

train_bcn_ft_loader = DataLoader(
    train_bcn_ft_ds,
    batch_size=BATCH_SIZE,
    shuffle=(train_bcn_ft_sampler is None),
    sampler=train_bcn_ft_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

val_bcn_ft_loader = DataLoader(
    val_bcn_ft_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

test_bcn_loader = DataLoader(
    test_bcn_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

print("DataLoaders created.")
print("train_base batches:", len(train_base_loader))
print("val_base batches:", len(val_base_loader))
print("train_bcn_ft batches:", len(train_bcn_ft_loader))
print("val_bcn_ft batches:", len(val_bcn_ft_loader))
print("test_bcn batches:", len(test_bcn_loader))


DataLoaders created.
train_base batches: 283
val_base batches: 71
train_bcn_ft batches: 301
val_bcn_ft batches: 102
test_bcn batches: 102


## Quick visual sanity check


In [18]:
sample_imgs, sample_targets = next(iter(train_base_loader))
print("Sample batch shape:", sample_imgs.shape)
print("Sample targets shape:", sample_targets.shape)
print("First 10 target indices:", sample_targets[:10].tolist())
print("First 10 target labels:", [IDX_TO_CLASS[int(i)] for i in sample_targets[:10]])


Sample batch shape: torch.Size([32, 3, 224, 224])
Sample targets shape: torch.Size([32])
First 10 target indices: [0, 0, 0, 0, 2, 0, 0, 1, 0, 3]
First 10 target labels: ['NV', 'NV', 'NV', 'NV', 'BCC', 'NV', 'NV', 'MEL', 'NV', 'BKL']


## Model setup


In [19]:
def build_model(model_name, num_classes, pretrained=True):
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=num_classes,
    )
    return model

model = build_model(MODEL_NAME, num_classes=len(PRIMARY_CLASSES), pretrained=True)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model.__class__.__name__)
print("Total params:", f"{n_params:,}")
print("Trainable params:", f"{n_trainable:,}")


ConvNeXt
Total params: 27,824,742
Trainable params: 27,824,742


## Training and evaluation helpers


In [20]:
def build_optimizer(model, lr, weight_decay):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def build_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

def build_loss(class_weights, label_smoothing=0.0):
    weight_tensor = torch.tensor(class_weights.values, dtype=torch.float32, device=DEVICE)
    return nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=label_smoothing)

def compute_metrics(y_true, y_pred, class_names):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    per_class_recall = recall_score(y_true, y_pred, average=None, labels=list(range(len(class_names))), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

    metrics = {
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(macro_f1),
        "per_class_recall": {class_names[i]: float(per_class_recall[i]) for i in range(len(class_names))},
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_true, y_pred,
            labels=list(range(len(class_names))),
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
    }
    return metrics

def train_one_epoch(model, loader, optimizer, criterion, device, epoch=None, stage_name="train"):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc=f"{stage_name} | epoch {epoch}", leave=False)

    for imgs, targets in pbar:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        avg_loss = running_loss / max(1, ((pbar.n + 1) * loader.batch_size))
        pbar.set_postfix(loss=f"{avg_loss:.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch=None, stage_name="eval"):
    model.eval()
    running_loss = 0.0
    all_targets = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{stage_name} | epoch {epoch}", leave=False)

    for batch in pbar:
        if len(batch) == 3:
            imgs, targets, _ = batch
        else:
            imgs, targets = batch

        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(imgs)
        loss = criterion(logits, targets)

        preds = logits.argmax(dim=1)

        running_loss += loss.item() * imgs.size(0)
        all_targets.extend(targets.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

        avg_loss = running_loss / max(1, ((pbar.n + 1) * loader.batch_size))
        pbar.set_postfix(loss=f"{avg_loss:.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_targets, all_preds, PRIMARY_CLASSES)
    metrics["loss"] = float(epoch_loss)
    return metrics

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    print("Saved json:", path)

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_metric, stage_name):
    ckpt = {
        "stage_name": stage_name,
        "epoch": epoch,
        "best_metric": best_metric,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "class_to_idx": CLASS_TO_IDX,
        "idx_to_class": IDX_TO_CLASS,
        "model_name": MODEL_NAME,
        "image_size": IMAGE_SIZE,
    }
    torch.save(ckpt, path)
    print("Saved checkpoint:", path)

def load_checkpoint_into_model(path, model, optimizer=None, scheduler=None, map_location=DEVICE):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model_state_dict"])

    if optimizer is not None and ckpt.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    return ckpt

def fit_stage(
    model,
    train_loader,
    val_loader,
    class_weights,
    lr,
    weight_decay,
    epochs,
    stage_name,
    checkpoint_path,
    metrics_json_path,
    label_smoothing=0.0,
):
    optimizer = build_optimizer(model, lr=lr, weight_decay=weight_decay)
    scheduler = build_scheduler(optimizer, epochs=epochs)
    criterion = build_loss(class_weights, label_smoothing=label_smoothing)

    history = []
    best_model_wts = copy.deepcopy(model.state_dict())
    best_bal_acc = -np.inf
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE,
            epoch=epoch, stage_name=f"{stage_name}_train"
        )

        val_metrics = evaluate(
            model, val_loader, criterion, DEVICE,
            epoch=epoch, stage_name=f"{stage_name}_val"
        )

        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_metrics["loss"]),
            "val_accuracy": float(val_metrics["accuracy"]),
            "val_balanced_accuracy": float(val_metrics["balanced_accuracy"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
        }
        history.append(row)

        print(
            f"[{stage_name}] Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["balanced_accuracy"] > best_bal_acc:
            best_bal_acc = val_metrics["balanced_accuracy"]
            best_epoch = epoch
            best_model_wts = copy.deepcopy(model.state_dict())

            save_checkpoint(
                checkpoint_path,
                model,
                optimizer,
                scheduler,
                epoch=epoch,
                best_metric=best_bal_acc,
                stage_name=stage_name,
            )

    model.load_state_dict(best_model_wts)

    final_criterion = build_loss(class_weights, label_smoothing=label_smoothing)
    best_val_metrics = evaluate(model, val_loader, final_criterion, DEVICE)

    out = {
        "stage_name": stage_name,
        "best_epoch": int(best_epoch),
        "best_balanced_accuracy": float(best_bal_acc),
        "history": history,
        "best_val_metrics": best_val_metrics,
    }

    save_json(out, metrics_json_path)
    return model, out

@torch.no_grad()
def predict_with_metadata(model, loader, device):
    model.eval()

    rows = []

    softmax = nn.Softmax(dim=1)

    for imgs, targets, metas in loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(imgs)
        probs = softmax(logits)
        preds = logits.argmax(dim=1)

        for i in range(imgs.size(0)):
            meta = {k: metas[k][i] for k in metas}
            row = {
                "isic_id": meta["isic_id"],
                "lesion_id": meta["lesion_id"],
                "image_path": meta["image_path"],
                "source_dataset": meta["source_dataset"],
                "y_true_idx": int(targets[i].item()),
                "y_true": IDX_TO_CLASS[int(targets[i].item())],
                "y_pred_idx": int(preds[i].item()),
                "y_pred": IDX_TO_CLASS[int(preds[i].item())],
            }
            for c_idx, c_name in IDX_TO_CLASS.items():
                row[f"prob_{c_name}"] = float(probs[i, c_idx].item())
            rows.append(row)

    return pd.DataFrame(rows)

def plot_confusion_matrix(cm, class_names, title, save_path):
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111)
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved figure:", save_path)


## Stage 1: HAM base training


In [21]:
base_model = build_model(MODEL_NAME, num_classes=len(PRIMARY_CLASSES), pretrained=True).to(DEVICE)

BASE_BEST_CKPT = CKPT_DIR / "base_best.pt"
BASE_METRICS_JSON = METRICS_DIR / "base_val_metrics.json"

base_model, base_out = fit_stage(
    model=base_model,
    train_loader=train_base_loader,
    val_loader=val_base_loader,
    class_weights=ham_weights,
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    epochs=BASE_EPOCHS,
    stage_name="base_ham",
    checkpoint_path=BASE_BEST_CKPT,
    metrics_json_path=BASE_METRICS_JSON,
    label_smoothing=LABEL_SMOOTHING,
)

print("\nBase stage best epoch:", base_out["best_epoch"])
print("Base stage best balanced accuracy:", base_out["best_balanced_accuracy"])


base_ham_train | epoch 1:   0%|          | 0/283 [00:00<?, ?it/s]

base_ham_val | epoch 1:   0%|          | 0/71 [00:00<?, ?it/s]

[base_ham] Epoch 01/8 | train_loss=2.3515 | val_loss=2.2094 | val_acc=0.0142 | val_bal_acc=0.1667 | val_macro_f1=0.0047
Saved checkpoint: ../outputs/checkpoints/base_best.pt


base_ham_train | epoch 2:   0%|          | 0/283 [00:00<?, ?it/s]

base_ham_val | epoch 2:   0%|          | 0/71 [00:00<?, ?it/s]

[base_ham] Epoch 02/8 | train_loss=2.1777 | val_loss=2.1463 | val_acc=0.5841 | val_bal_acc=0.2554 | val_macro_f1=0.1843
Saved checkpoint: ../outputs/checkpoints/base_best.pt


base_ham_train | epoch 3:   0%|          | 0/283 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Review Stage 1 validation metrics


In [ ]:
display(pd.DataFrame(base_out["history"]))
print("Best validation metrics (HAM val):")
print(json.dumps({
    "accuracy": base_out["best_val_metrics"]["accuracy"],
    "balanced_accuracy": base_out["best_val_metrics"]["balanced_accuracy"],
    "macro_f1": base_out["best_val_metrics"]["macro_f1"],
    "per_class_recall": base_out["best_val_metrics"]["per_class_recall"],
}, indent=2))


## Stage 2: BCN fine-tuning

We start from the best HAM checkpoint and adapt the model to the BCN target domain.


In [ ]:
ft_model = build_model(MODEL_NAME, num_classes=len(PRIMARY_CLASSES), pretrained=False).to(DEVICE)
_ = load_checkpoint_into_model(BASE_BEST_CKPT, ft_model, optimizer=None, scheduler=None, map_location=DEVICE)

FT_BEST_CKPT = CKPT_DIR / "bcn_ft_best.pt"
FT_METRICS_JSON = METRICS_DIR / "bcn_val_metrics.json"

ft_model, ft_out = fit_stage(
    model=ft_model,
    train_loader=train_bcn_ft_loader,
    val_loader=val_bcn_ft_loader,
    class_weights=bcn_weights,
    lr=FT_LR,
    weight_decay=WEIGHT_DECAY,
    epochs=FT_EPOCHS,
    stage_name="finetune_bcn",
    checkpoint_path=FT_BEST_CKPT,
    metrics_json_path=FT_METRICS_JSON,
    label_smoothing=LABEL_SMOOTHING,
)

print("\nFine-tune stage best epoch:", ft_out["best_epoch"])
print("Fine-tune stage best balanced accuracy:", ft_out["best_balanced_accuracy"])


## Review Stage 2 validation metrics


In [ ]:
display(pd.DataFrame(ft_out["history"]))
print("Best validation metrics (BCN val):")
print(json.dumps({
    "accuracy": ft_out["best_val_metrics"]["accuracy"],
    "balanced_accuracy": ft_out["best_val_metrics"]["balanced_accuracy"],
    "macro_f1": ft_out["best_val_metrics"]["macro_f1"],
    "per_class_recall": ft_out["best_val_metrics"]["per_class_recall"],
}, indent=2))


## Final evaluation on BCN test


In [ ]:
best_ft_model = build_model(MODEL_NAME, num_classes=len(PRIMARY_CLASSES), pretrained=False).to(DEVICE)
_ = load_checkpoint_into_model(FT_BEST_CKPT, best_ft_model, optimizer=None, scheduler=None, map_location=DEVICE)

test_criterion = build_loss(bcn_weights, label_smoothing=0.0)
test_metrics = evaluate(best_ft_model, test_bcn_loader, test_criterion, DEVICE)

TEST_METRICS_JSON = METRICS_DIR / "test_bcn_metrics.json"
save_json(test_metrics, TEST_METRICS_JSON)

print("Test metrics:")
print(json.dumps({
    "accuracy": test_metrics["accuracy"],
    "balanced_accuracy": test_metrics["balanced_accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "per_class_recall": test_metrics["per_class_recall"],
}, indent=2))


## Confusion matrix on BCN test


In [ ]:
test_cm = np.array(test_metrics["confusion_matrix"])
CONF_MAT_PATH = FIG_DIR / "test_bcn_confusion_matrix.png"

plot_confusion_matrix(
    test_cm,
    class_names=PRIMARY_CLASSES,
    title="BCN Test Confusion Matrix",
    save_path=CONF_MAT_PATH
)


## Per-image predictions on BCN test

This is useful for:
- confusion analysis
- selecting images for CAM panels
- later comparison of correct vs incorrect cases


In [ ]:
test_pred_df = predict_with_metadata(best_ft_model, test_bcn_loader, DEVICE)
TEST_PRED_CSV = PRED_DIR / "test_bcn_predictions.csv"
test_pred_df.to_csv(TEST_PRED_CSV, index=False)

print("Saved test predictions:", TEST_PRED_CSV)
display(test_pred_df.head())


## Quick error analysis


In [ ]:
test_pred_df["correct"] = (test_pred_df["y_true"] == test_pred_df["y_pred"])

print("Overall correctness:")
display(test_pred_df["correct"].value_counts(dropna=False).to_frame("count"))

print("\nMost common confusions:")
confusions = (
    test_pred_df.loc[~test_pred_df["correct"]]
    .groupby(["y_true", "y_pred"])
    .size()
    .sort_values(ascending=False)
    .to_frame("count")
    .reset_index()
)
display(confusions.head(20))


## Save a compact experiment summary


In [ ]:
summary = {
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "device": DEVICE,
    "base_epochs": BASE_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "base_lr": BASE_LR,
    "ft_lr": FT_LR,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "classes": PRIMARY_CLASSES,
    "base_best_epoch": base_out["best_epoch"],
    "base_best_balanced_accuracy": base_out["best_balanced_accuracy"],
    "ft_best_epoch": ft_out["best_epoch"],
    "ft_best_balanced_accuracy": ft_out["best_balanced_accuracy"],
    "test_accuracy": test_metrics["accuracy"],
    "test_balanced_accuracy": test_metrics["balanced_accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_per_class_recall": test_metrics["per_class_recall"],
}

SUMMARY_JSON = METRICS_DIR / "baseline_experiment_summary.json"
save_json(summary, SUMMARY_JSON)
print(json.dumps(summary, indent=2))


## What to do next

If this baseline works, the next step is:

### baseline CAM generation
Use:
- BCN test predictions
- the saved best BCN fine-tuned checkpoint
- a fixed subset of:
  - correct cases
  - hard cases
  - common confusions

Then compare:
- Grad-CAM
- LayerCAM
- difference-CAM

Only after that should you add:
- saliency separation loss
- optional mask-based localization regularization
